In [129]:
import numpy as np

class Softmax:
    # Xavier initialization
    weight_init_factor = 1.

    def forward(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

class Sigmoid:
    # Xavier initialization
    weight_init_factor = 1.

    def forward(self, x):
        return 1 / (1 + np.exp(-x))
    
    def derivative(self, x):
        s = self.forward(x)
        return s * (1 - s)

class ReLU:
    # He initialization
    weight_init_factor = np.sqrt(2)

    def forward(self, x):
        return np.maximum(0, x)
    
    def derivative(self, x):
        return (x > 0).astype(float)

class Linear:
    # Xavier initialization
    weight_init_factor = 1.0

    def forward(self, x):
        return x

    def derivative(self, x):
        return np.ones_like(x)

In [130]:
class Layer:
    def __init__(self, num_neurons: int, num_input_connections: int, activation_function, rng, alpha=0.05, beta_one=0.9, beta_two=0.999, epsilon = 1e-8):
        self.num_neurons = num_neurons
        self.activation_function = activation_function
        self.weights =  rng.standard_normal((num_input_connections, num_neurons)) * activation_function.weight_init_factor * np.sqrt(1/num_input_connections)
        self.biases = self.biases = rng.standard_normal(num_neurons) * 0.01
        self.prev_inputs = None
        self.z = None
        self.error_signals = None
        self.alpha = alpha


        # adam optimizer stuff
        self.ma_weight_signal = np.zeros((num_input_connections, num_neurons)) # moving average bias gradient signal for adam optimizer
        self.ma_bias_signal = np.zeros(num_neurons) # moving average weight gradient signal for adam optimizer
        self.update_steps = 0
        self.beta_one = beta_one
        self.ma_weight_signal_variance = np.zeros((num_input_connections, num_neurons))
        self.ma_bias_signal_variance = np.zeros(num_neurons)
        self.beta_two = beta_two
        self.epsilon = epsilon

    def forward(self, x) -> np.array:
        self.z = x @ self.weights + self.biases
        self.prev_inputs = x
        return self.activation_function.forward(self.z)

    def backward(self, error_signals) -> np.array:
        self.update_steps += 1

        # i didnt add a derivative to softmax cus im lazy, i didn't feel like spending the time figuring out the jacobians, just passing the loss for the next step all the way through
        if hasattr(self.activation_function, 'derivative'):
            dLdz = error_signals * self.activation_function.derivative(self.z)
        else:
            dLdz = error_signals
    
        batch_size = error_signals.shape[0]

        bias_gradient_signal = np.sum(dLdz, axis=0) / batch_size
        self.ma_bias_signal = (self.beta_one * self.ma_bias_signal) + (1 - self.beta_one) * bias_gradient_signal # first moment
        self.ma_bias_signal_variance = (self.beta_two * self.ma_bias_signal_variance) + (1 - self.beta_two) * bias_gradient_signal ** 2 # second moment
        mb = self.ma_bias_signal / (1 - self.beta_one ** self.update_steps) # bias correction
        vb = self.ma_bias_signal_variance / (1 - self.beta_two ** self.update_steps) # bias correction
        dB = mb / (np.sqrt(vb) + self.epsilon)

        weight_gradient_signal = (self.prev_inputs.T @ dLdz) / batch_size
        self.ma_weight_signal = ( (self.beta_one * self.ma_weight_signal) + (1 - self.beta_one) * weight_gradient_signal ) # first moment
        self.ma_weight_signal_variance = (self.beta_two * self.ma_weight_signal_variance) + (1 - self.beta_two) * weight_gradient_signal ** 2 # second moment
        mw = self.ma_weight_signal / (1 - self.beta_one ** self.update_steps) # bias correction
        vw = self.ma_weight_signal_variance / (1 - self.beta_two ** self.update_steps) # bias correction
        dW = mw / (np.sqrt(vw) + self.epsilon)

        error_signals = dLdz @ self.weights.T
        self.biases += -self.alpha * dB
        self.weights += -self.alpha * dW

        return error_signals

In [131]:
import numpy as np


def load_model(layers, filepath="mnist_model.npz"):
    data = np.load(filepath)
    for i, layer in enumerate(layers):
        layer.weights = data[f"w_{i}"]
        layer.biases = data[f"b_{i}"]


pretrained_classifier = [
    Layer(
        num_neurons=256,
        num_input_connections=784,
        activation_function=ReLU(),
        rng=np.random.default_rng(),
    ),
    Layer(
        num_neurons=10,
        num_input_connections=256,
        activation_function=Softmax(),
        rng=np.random.default_rng(),
    ),
]

load_model(pretrained_classifier, "mnist_model.npz")

In [132]:
%matplotlib inline
import matplotlib.pyplot as plt
from os.path import join
import struct

#
# MNIST Data Loader Class
#
class MnistDataloader(object):
    def __init__(self, training_images_filepath, training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError(f'Magic number mismatch, expected 2049, got {magic}')
            labels = np.frombuffer(file.read(), dtype=np.uint8)
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError(f'Magic number mismatch, expected 2051, got {magic}')
            images = np.frombuffer(file.read(), dtype=np.uint8).reshape(size, rows, cols)
        
        return images, labels
            
    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train), (x_test, y_test)

#
# Set file paths based on added MNIST Datasets
#
input_path = ''
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

#
# Helper function to show a list of images with their relating titles
#
def show_images(images, title_texts):
    cols = 5
    rows = int(len(images)/cols) + 1
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image, cmap=plt.cm.gray)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1

#
# Load MINST dataset
#
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
(x_train, y_train), (x_test, y_test) = mnist_dataloader.load_data()


x_train = np.array([x.ravel() for x in x_train], dtype=np.float32) / 255.0
x_test = np.array([x.ravel() for x in x_test], dtype=np.float32) / 255.0

def get_batches(X, y, batch_size):
    num_samples = X.shape[0]
    indices = np.arange(num_samples)
    
    np.random.shuffle(indices)
    
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)
        batch_indices = indices[start_idx:end_idx]
        
        yield X[batch_indices], y[batch_indices]

print(x_train.shape)
print(x_test.shape)



(60000, 784)
(10000, 784)


In [133]:
rng = np.random.default_rng(42)


buffer_capacity = 100_000
state_size = 784 + 784 + 1   # 784 values for the current observed values of all states, 
                             # 784 values in the mask (which pixels we currently have access to), 
                             # and 1 pixel for the remaining budget of pixel reveals left)
num_pixel_reveals = 60
epsilon = 0.9 # for epsilon greedy search

class Network():
    def __init__(self):
        self.layers = [
            Layer(num_neurons=256, num_input_connections=state_size, activation_function=ReLU(), rng=rng, alpha=0.001),
            Layer(num_neurons=128, num_input_connections=256, activation_function=ReLU(), rng=rng, alpha=0.001),
            Layer(num_neurons=784, num_input_connections=128, activation_function=Linear(), rng=rng, alpha=0.001)
        ]

    def full_forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def full_backward(self, upstream_error):
        for layer in reversed(self.layers):
            upstream_error = layer.backward(upstream_error)
        return upstream_error

    def copy_weights_from(self, network: Network):
        # not copying any of the adam optimizer stuff bcs this is for the target network so its never doing gradient descent
        for i, layer in enumerate(self.layers):
            layer.weights = network.layers[i].weights.copy()
            layer.biases = network.layers[i].biases.copy()


In [134]:
discount_factor = 1
buffer_s = np.zeros((buffer_capacity, state_size))
buffer_a = np.zeros(buffer_capacity, dtype=np.int32)
buffer_r = np.zeros(buffer_capacity)
buffer_s_prime = np.zeros((buffer_capacity, state_size))
buffer_dones = np.zeros(buffer_capacity)

buffer_ptr = 0 # points to the index of where weare adding to the buffer
buffer_size = 0 # tracks the current size of the buffer

def reveal_pixel(s, a, secret_images):
    s_copy = s.copy()
    env_indices = np.arange(s.shape[0])
    s_copy[env_indices, a] = secret_images[env_indices, a]
    s_copy[env_indices, a+784] = 1
    s_copy[env_indices, -1] -= 1/num_pixel_reveals
    return s_copy # replace with revealing one pixel from MNIST image

def get_reward(state, secret_labels):
    out = state[:, :784]
    for layer in pretrained_classifier:
        out = layer.forward(out)
    predictions = np.argmax(out, axis=1)
    return predictions==secret_labels * 2 - 1 # simple reward +1 if it guesses correctly and -1 if it guesses wrong

def add_to_buffer(s, a, r, s_prime, done):
    global buffer_ptr, buffer_size
    num_samples = s.shape[0]
    indicies = np.arange(buffer_ptr, buffer_ptr + num_samples) % buffer_capacity

    buffer_s[indicies] = s
    buffer_a[indicies] = a
    buffer_r[indicies] = r
    buffer_s_prime[indicies] = s_prime
    buffer_dones[indicies] = done

    buffer_ptr = (buffer_ptr + num_samples) % buffer_capacity
    buffer_size = min(buffer_size + num_samples, buffer_capacity)

def interact(network, num_envs=32):
    global x_train
    s = np.zeros((num_envs, state_size))
    s[:, -1] = num_pixel_reveals/num_pixel_reveals
    secret_image_indicies = rng.integers(low=0, high=60_000, size=num_envs) # indicies of rnadom integers
    secret_images = x_train[secret_image_indicies]
    secret_labels = y_train[secret_image_indicies]


    for pixel_reveals_left in range(num_pixel_reveals, 0, -1):
        q_values = network.full_forward(s) # shape = (num_envs, num actions)
        mask_prev_actions = s[:, q_values.shape[1] : -1].astype(bool) # mask so the agen't doesn't try to reveal a pixel that it has already revealed
        assert np.all(np.sum(mask_prev_actions, axis=1) < 784)
        greedy_actions = np.argmax(np.where(mask_prev_actions, float('-inf'), q_values), axis=1)  # shape = (num_envs,)

        mask_epsilon = rng.random(size=num_envs) < epsilon
        uniform_random = np.where(mask_prev_actions, float('-inf'), rng.random((num_envs, 784))) # mask epsilon for not picking an action we have already took
        random_actions = np.argmax(uniform_random, axis=1)

        a = np.where(mask_epsilon, random_actions, greedy_actions) # apply the mask (epsilon greedy search)
        s_prime = reveal_pixel(s, a, secret_images)

        if pixel_reveals_left == 1:
            s_prime[:, -1] = 0
            add_to_buffer(s, a, get_reward(s_prime, secret_labels), s_prime, np.ones(num_envs))
        else:
            add_to_buffer(s, a, np.zeros(num_envs), s_prime, np.zeros(num_envs))
        s = s_prime



In [ ]:
import sys

batch_size = 128

online_network = Network()
target_network = Network()
target_network.copy_weights_from(online_network)

# warm up the buffer
while buffer_size < 20000:
    interact(online_network)

grad_steps = 0
for iteration in range(10000):

    interact(online_network) # interacts with the environment and populates the replay buffer

    for _ in range(15): # 15 gradient steps
        indices = rng.choice(buffer_size, size=batch_size, replace=False)
        batch_s = buffer_s[indices]
        batch_a = buffer_a[indices]
        batch_r = buffer_r[indices]
        batch_s_prime = buffer_s_prime[indices]
        batch_dones = buffer_dones[indices]

        q_values_s = online_network.full_forward(batch_s)
        q_sa = q_values_s[np.arange(batch_size), batch_a] # what was the q value for the action we chose
        loss_grad =  q_sa - ( batch_r + (1-batch_dones) * discount_factor * np.max(target_network.full_forward(batch_s_prime), axis=1) )

        # the mlp needs gradients of shape = (batch size, amount of mlp outputs)
        filled_grads = np.zeros((batch_size, 784))
        filled_grads[np.arange(batch_size), batch_a] = loss_grad

        online_network.full_backward(filled_grads)
        grad_steps += 1

    if grad_steps % 200 == 0:
        target_network.copy_weights_from(online_network)

    epsilon = max(0.05, epsilon * 0.9995)
    sys.exit()

SystemExit: 

c:\Users\yhtru\OneDrive\Desktop\dev\physical_ml\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
